# StreamStay: Notebook 01 — ML System Lifecycle & Skew Prevention (M1)
## End-to-End Prediction Architecture & Shared scikit-learn Pipeline

This notebook demonstrates:
1. Generating 20,000 synthetic subscribers with 15% churn rate and ~5% injected missingness.
2. Building the shared scikit-learn ColumnTransformer (median imputation, MissingIndicator, scaling, one-hot encoding).
3. Tracing a prediction from raw JSON payload to inference to eliminate training-serving skew.


In [ ]:
import sys, os
import pandas as pd
import numpy as np

if os.path.basename(os.getcwd()) == "notebooks":
    os.chdir("..")

from src.data_generator import generate_streamstay_dataset
from src.features import build_preprocessor, split_streamstay_data, NUMERICAL_FEATURES, CATEGORICAL_FEATURES

df = generate_streamstay_dataset(n_samples=20000, random_seed=42, save_path="data/streamstay_subscribers.csv")
print("Dataset Shape:", df.shape)
print("Churn Rate:", f"{df['churned'].mean():.2%}")
df.head()


In [ ]:
# Fit preprocessor on training partition only to prevent leakage
splits = split_streamstay_data(df, test_size=0.15, val_size=0.15, random_seed=42)
preproc = build_preprocessor(include_missing_indicator=True)
preproc.fit(splits["train"][NUMERICAL_FEATURES + CATEGORICAL_FEATURES])
print("Fitted ColumnTransformer successfully. Zero training-serving skew.")
